# 02 · Scheduled route and hourly supply

**Purpose:** describe how many trips the GTFS schedules by route and departure hour for one selected service date. These are supply metrics, not ridership observations.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.gtfs import resolve_gtfs_dir, load_gtfs

GTFS_DIR = resolve_gtfs_dir(PROJECT_ROOT)
feed = load_gtfs(GTFS_DIR)
print(f"Feed folder: {GTFS_DIR}")
from src.gtfs import active_service_ids, first_trip_departures

# Edit this to a date covered by calendar.txt. This feed covers 2024-05-01 through 2030-05-01.
SERVICE_DATE = "2026-10-26"  # Monday
active_ids = active_service_ids(feed["calendar"], SERVICE_DATE)
print(f"Active service IDs on {SERVICE_DATE}: {len(active_ids)}")
if not active_ids:
    raise ValueError("No calendar service is active on SERVICE_DATE; choose a date within the feed coverage.")

## 1. Count scheduled trips by route and departure hour

In [ ]:
departures = first_trip_departures(feed["stop_times"])
trips = feed["trips"].copy()
trips["service_id"] = trips["service_id"].astype(str)
trips = trips[trips["service_id"].isin(active_ids)].merge(departures, on="trip_id", how="left", validate="one_to_one")
trips = trips.merge(feed["routes"], on="route_id", how="left", validate="many_to_one")
trips = trips.dropna(subset=["departure_hour"])
trips["departure_hour"] = trips["departure_hour"].astype(int)

route_hour_supply = (trips.groupby(["route_id", "route_short_name", "route_long_name", "departure_hour"], dropna=False)
                     .size().rename("scheduled_trip_count").reset_index())
print(f"Scheduled trips counted: {len(trips):,}")
display(route_hour_supply.head(10))

## 2. Route summary and supply chart

In [ ]:
route_summary = (route_hour_supply.groupby(["route_id", "route_short_name", "route_long_name"], dropna=False)
                 .agg(scheduled_trips=("scheduled_trip_count", "sum"),
                      operating_departure_hours=("departure_hour", "nunique"))
                 .reset_index().sort_values("scheduled_trips", ascending=False))
display(route_summary.head(20))

hourly_supply = route_hour_supply.groupby("departure_hour")["scheduled_trip_count"].sum().reindex(range(24), fill_value=0)
ax = hourly_supply.plot(kind="bar", figsize=(12, 4), color="#247ba0", title=f"Scheduled trip departures by hour · {SERVICE_DATE}")
ax.set_xlabel("Departure hour (local feed time)")
ax.set_ylabel("GTFS scheduled trips")
ax.grid(axis="y", alpha=.25)


## 3. Export schedule features

The CSV is generated locally and is ignored by Git. It contains schedule-derived features only.

In [ ]:
output_dir = PROJECT_ROOT / "dataset" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)
supply_path = output_dir / f"chennai_route_hour_supply_{SERVICE_DATE}.csv"
route_hour_supply.to_csv(supply_path, index=False)
print(f"Wrote {supply_path}")

## Interpretation

A route with more scheduled trips has more service supply in the feed. It does not follow that it has more riders or is overcrowded. Use the supply table as a baseline for comparing a later ridership dataset or as the allocation basis for the explicitly synthetic exercise in notebook 03.